# Demo 5 — Confrontare comb FIR e IIR nel tempo e in frequenza

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
D, g = 220, 0.7
b_comb = np.zeros(D+1)
b_comb[0], b_comb[-1] = 1., g
a_comb = np.zeros(D+1)
a_comb[0], a_comb[-1] = 1., -g
imp_comb = np.zeros(10*D+1)
imp_comb[0] = 1.
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Risposte all'impulso", "Risposte in frequenza"])
for nome, b_c, a_c in (("Comb FIR", b_comb, [1.]), ("Comb IIR", [1.], a_comb)):
    h_c = al.filter_signal(b_c, a_c, imp_comb)
    f_c, H_c = signal.freqz(b_c, a_c, worN=16384, fs=SR)
    fig.add_trace(go.Scatter(x=np.arange(len(h_c))/SR*1000, y=h_c,
                            name=nome), row=1, col=1)
    fig.add_trace(go.Scatter(x=f_c, y=al.to_db(np.abs(H_c), floor=1e-6), name=nome,
                            showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Tempo [ms]", row=1, col=1)
fig.update_yaxes(title_text="Ampiezza", row=1, col=1)
fig.update_xaxes(title_text="Frequenza [Hz]", range=[0, 2000], row=1, col=2)
fig.update_yaxes(title_text="Guadagno [dB]", row=1, col=2)
fig.update_layout(height=380)
fig.show()
print(f"Ritardo: {1000*D/SR:.3f} ms; distanza fra picchi: {SR/D:.2f} Hz")
# Si raccolgono 20 intervalli di coda; l'IIR resta idealmente infinito.
chitarra_c, sr = al.load_audio(AUDIO / "Guitar_C4.wav", sr=SR, seconds=4.0)
x_c = np.pad(chitarra_c, (0, 20*D))
al.compare_audio([("Originale", x_c),
                  ("Comb FIR", al.filter_signal(b_comb, [1.], x_c)),
                  ("Comb IIR", al.filter_signal([1.], a_comb, x_c))], sr)
